# 1. Onboarding: identities, credentials and peer tokens

Before two connectors can speak the Dataspace Protocol (DSP 2025-1) they must know who the other
one is and hold a token to present on every DSP call. This notebook onboards the three
participants of the dev dataspace using the Auth Management API (`/api/v1/...`) and the Well-Known
RPCs (`/rpc/.well-known/...`):

| Participant | Default URL | Role |
|---|---|---|
| Authority | `http://127.0.0.1:1500` | Issues the `DataSpaceParticipant` verifiable credential |
| Consumer | `http://127.0.0.1:1100` | Discovers the catalog, negotiates and receives data |
| Provider | `http://127.0.0.1:1200` | Publishes datasets, agrees contracts and serves data |

Steps:
1. Configuration, HTTP helpers and lightweight demo backend/sink servers for transfers.
2. Link and inspect the wallet of each participant (`/api/v1/wallet/is-linked`, `/api/v1/wallet/link`, `/api/v1/wallet/info`).
3. Read each participant's DID (`/.well-known/did.json`) and self-registration (`/api/v1/mates/myself`).
4. Consumer and Provider obtain a credential from the Authority (`/api/v1/vc-request/beg`, `/api/v1/vc-request/all`, `/api/v1/wallet/vcs`).
5. Consumer and Provider connect to each other and exchange peer tokens (`/api/v1/peer-connection/connect`, `/api/v1/peer-connection/request/all`, `/api/v1/mates/all`).
6. Resolve the Consumer and Provider DSP base URLs (`/.well-known/dspace-version/2025-1` and `/rpc/.well-known/dspace-version/path`).

**Runs on its own.** Every other notebook runs it first through a guarded `%run`, so it executes
once per kernel. All onboarding steps are idempotent.

## 1. Configuration and helpers

The URLs can be overridden with environment variables. The `INTERNAL_*` variants are the addresses
the agents use to reach each other (they differ from the public ones when the agents run inside
Docker).

`show(response)` prints an HTTP answer as JSON and fails the cell on a non-2xx status, so a broken
step stops a chained `%run` instead of feeding empty ids to the next one. Pass `ok=False` when a
failure is the expected result.

`ensure_demo_servers()` starts lightweight in-kernel HTTP listeners on ports `8081` (PULL source),
`8082` (PUSH subscription backend) and `1112` (Consumer PUSH sink) whenever those ports are not
already bound by an external service.

In [1]:
import json
import os
import socket
import threading
import time
from http.server import BaseHTTPRequestHandler, ThreadingHTTPServer

import requests

data_space_authority = os.environ.get("AUTHORITY_URL", "http://127.0.0.1:1500")
data_space_consumer = os.environ.get("CONSUMER_URL", "http://127.0.0.1:1100")
data_space_provider = os.environ.get("PROVIDER_URL", "http://127.0.0.1:1200")

internal_authority_url = os.environ.get("INTERNAL_AUTHORITY_URL", data_space_authority)
internal_consumer_url = os.environ.get("INTERNAL_CONSUMER_URL", data_space_consumer)
internal_provider_url = os.environ.get("INTERNAL_PROVIDER_URL", data_space_provider)

json_header = {"Content-Type": "application/json"}

# JSON-LD context of every Eunomia RPC body (negotiation and transfer)
eunomia_context = "https://w3id.org/eunomia/v1/context.jsonld"


def show(response, ok=True):
    """Prints an HTTP answer as JSON; with ok, raises unless the status is 2xx."""
    try:
        body = response.json()
    except ValueError:
        body = response.text
    print(f"HTTP {response.status_code} {response.request.method} {response.url}")
    print(body if isinstance(body, str) else json.dumps(body, indent=2))
    if ok and not response.ok:
        raise RuntimeError(f"{response.request.method} {response.url} answered {response.status_code}")
    return body


demo_sink_events = []
demo_subscriptions = []


class _DemoHandler(BaseHTTPRequestHandler):
    def log_message(self, fmt, *args):
        pass

    def _send_json(self, status, payload):
        raw = json.dumps(payload).encode("utf-8")
        self.send_response(status)
        self.send_header("Content-Type", "application/json")
        self.send_header("Content-Length", str(len(raw)))
        self.end_headers()
        self.wfile.write(raw)

    def _read_json(self):
        length = int(self.headers.get("Content-Length") or 0)
        if not length:
            return {}
        raw = self.rfile.read(length)
        try:
            return json.loads(raw.decode("utf-8"))
        except ValueError:
            return {"raw": raw.decode("utf-8", errors="replace")}

    def do_HEAD(self):
        self.send_response(200)
        self.send_header("Content-Type", "application/json")
        self.end_headers()

    def do_GET(self):
        self._send_json(200, {
            "dataset": "ecostars-hotel-sustainability",
            "path": self.path,
            "authHeaderPresent": bool(self.headers.get("Authorization")),
            "records": [
                {"hotel": "Madrid Central", "kwhPerNight": 14.2},
                {"hotel": "Atocha Suites", "kwhPerNight": 11.8},
            ],
        })

    def do_POST(self):
        body = self._read_json()
        if self.path.startswith("/subscriptions/subscribe"):
            sub_id = f"sub-{len(demo_subscriptions) + 1}"
            demo_subscriptions.append({"id": sub_id, "action": "subscribe", "body": body})
            self._send_json(200, {"data": {"ID": sub_id}, "subscription": body})
        elif self.path.startswith("/subscriptions/unsubscribe"):
            sub_id = self.path.rsplit("/", 1)[-1]
            demo_subscriptions.append({"id": sub_id, "action": "unsubscribe"})
            self._send_json(200, {"unsubscribed": sub_id})
        else:
            demo_sink_events.append({
                "path": self.path,
                "authorization": self.headers.get("Authorization"),
                "body": body,
            })
            self._send_json(200, {"received": True, "count": len(demo_sink_events)})


def ensure_demo_servers(ports=(8081, 8082, 1112)):
    """Starts background HTTP servers on any of `ports` not already listening on 127.0.0.1."""
    started = globals().setdefault("_demo_servers", {})
    for port in ports:
        if port in started:
            continue
        with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
            if s.connect_ex(("127.0.0.1", port)) == 0:
                continue
        srv = ThreadingHTTPServer(("127.0.0.1", port), _DemoHandler)
        thread = threading.Thread(target=srv.serve_forever, daemon=True)
        thread.start()
        started[port] = srv


ensure_demo_servers()

print(f"Authority URL: {data_space_authority}")
print(f"Consumer URL:  {data_space_consumer}")
print(f"Provider URL:  {data_space_provider}")

Authority URL: http://127.0.0.1:1500
Consumer URL:  http://127.0.0.1:1100
Provider URL:  http://127.0.0.1:1200


## 2. Link and inspect wallets (`/api/v1/wallet/*`)

Each agent checks `/api/v1/wallet/is-linked` and links its wallet via `POST /api/v1/wallet/link` if
needed, creating its keys and publishing its DID document. `GET /api/v1/wallet/info` returns the
wallet metadata and registered DIDs.

In [2]:
for name, agent_url in [("Authority", data_space_authority), ("Consumer", data_space_consumer), ("Provider", data_space_provider)]:
    linked = requests.get(f"{agent_url}/api/v1/wallet/is-linked", headers=json_header)
    if not linked.ok:
        res = requests.post(f"{agent_url}/api/v1/wallet/link", headers=json_header, json={})
        if res.status_code not in (200, 204):
            print(f"Wallet of {name} link status ({res.status_code}): {res.text}")
    info = requests.get(f"{agent_url}/api/v1/wallet/info", headers=json_header).json()
    print(f"Wallet linked for {name} ({agent_url}): wallet id={info.get('id')}, name={info.get('name')}")

Wallet linked for Authority (http://127.0.0.1:1500): wallet id=fafnir-local, name=fafnir-wallet
Wallet linked for Consumer (http://127.0.0.1:1100): wallet id=fafnir-local, name=fafnir-wallet
Wallet linked for Provider (http://127.0.0.1:1200): wallet id=fafnir-local, name=fafnir-wallet


## 3. Participant DIDs (`/.well-known/did.json` and `/api/v1/mates/myself`)

A participant is identified in the dataspace by its decentralized identifier (DID), published at
`/.well-known/did.json` and mirrored in the agent's own participant record at
`GET /api/v1/mates/myself`. The DSP and RPC calls below name the peer by this DID in
`associatedAgentPeer`.

In [3]:
data_space_authority_participant_id = requests.get(f"{data_space_authority}/.well-known/did.json").json()["id"]
data_space_consumer_participant_id = requests.get(f"{data_space_consumer}/.well-known/did.json").json()["id"]
data_space_provider_participant_id = requests.get(f"{data_space_provider}/.well-known/did.json").json()["id"]

consumer_self = show(requests.get(f"{data_space_consumer}/api/v1/mates/myself", headers=json_header))
assert consumer_self["participant_id"] == data_space_consumer_participant_id

print(f"\nAuthority DID: {data_space_authority_participant_id}\n")
print(f"Consumer DID:  {data_space_consumer_participant_id}\n")
print(f"Provider DID:  {data_space_provider_participant_id}")

HTTP 200 GET http://127.0.0.1:1100/api/v1/mates/myself
{
  "participant_id": "did:jwk:eyJlIjoiQVFBQiIsImt0eSI6IlJTQSIsIm4iOiJ5cS12NXdJNFM2MTl2SnJITmpENDVra0owUGw4VGlBTzZYTnFJc1o4SDBaTEs5T2JoWTBTeGtEQ2JlZWd2RlZEU0MzM182N3lVZVpUWFI2SjFXUExHdHNtLU51SXJtR1RVRVgteFA0WnFLVGRFUTFSdUV0MExtMUhtdkdJWG1zLVJrTXRNVjcxRWt0MjNiZFYzX01WTjRUOUd5WnlaQTB6RDRlTFUxRTRTRzlRMGl4MGFRbC16eGdsVFRYQ2l4VTdCTzRXcHRkaUp3a0Jid2hMbnVqeXFzSmNvQXpPU000aXZtZlRTVW1Yd0Nrc1gySTFxVTNnVWdtUGFtbmEta3ZUb3FYZUk4ODNlRTFkSWNuUG53dmJlRmpfa0xuQ2JrNzhjNkVkRXZ0OVBMa3ByTWZfdmVxeVlOWjB0WlpYUGM5c0lRVlMyUG5xZkN5RUhsWWVuMXNXclEifQ",
  "participant_nick": "Myself",
  "participant_type": "Agent",
  "base_url": "http://127.0.0.1:1100",
  "saved_at": "2026-10-26T20:22:24.040889Z",
  "last_interaction": "2026-10-26T20:22:24.040889Z",
  "extra_fields": {}
}

Authority DID: did:web:127.0.0.1%3A1500

Consumer DID:  did:jwk:eyJlIjoiQVFBQiIsImt0eSI6IlJTQSIsIm4iOiJ5cS12NXdJNFM2MTl2SnJITmpENDVra0owUGw4VGlBTzZYTnFJc1o4SDBaTEs5T2JoWTBTeGtEQ2JlZWd2RlZEU

## 4. Credentials from the Authority (`/api/v1/vc-request/*` and `/api/v1/wallet/vcs`)

Consumer and Provider ask the Authority for a `DataSpaceParticipant_jwt_vc_json` credential
through `POST /api/v1/vc-request/beg` (which contacts the Authority's gate at
`/api/v1/gate/access`). With `auto: true` the Authority issues it at once. A participant that
already holds a finalized credential in `GET /api/v1/vc-request/all` skips the request. The stored
verifiable credentials can also be listed via `GET /api/v1/wallet/vcs`.

In [4]:
vc_type = "DataSpaceParticipant_jwt_vc_json"


def has_credential(agent_url):
    """Whether the agent holds a finalized credential of vc_type from the Authority."""
    items = requests.get(f"{agent_url}/api/v1/vc-request/all", headers=json_header).json().get("items", [])
    return any(
        req.get("participant_id") == data_space_authority_participant_id
        and req.get("status") == "Finalized"
        and vc_type in (req.get("vc_type_config") or [])
        for req in items
    )


def request_credential(agent_name, agent_url):
    if has_credential(agent_url):
        vcs = requests.get(f"{agent_url}/api/v1/wallet/vcs", headers=json_header).json()
        print(f"{agent_name} already holds {vc_type} from the Authority ({len(vcs)} VCs in wallet).")
        return
    beg_body = {
        "url": f"{internal_authority_url}/api/v1/gate/access",
        "id": data_space_authority_participant_id,
        "nick": "authority",
        "vc_type": vc_type,
        "method": "cert",
        "auto": True,
    }
    res = requests.post(f"{agent_url}/api/v1/vc-request/beg", json=beg_body, headers=json_header)
    if res.status_code not in (200, 201, 204):
        raise RuntimeError(f"{agent_name} could not request a credential: {res.status_code} - {res.text}")
    if not has_credential(agent_url):
        raise RuntimeError(f"Credential issuance for {agent_name} did not finalize.")
    vcs = requests.get(f"{agent_url}/api/v1/wallet/vcs", headers=json_header).json()
    print(f"{agent_name} obtained {vc_type} from the Authority ({len(vcs)} VCs in wallet).")


request_credential("Consumer", data_space_consumer)
request_credential("Provider", data_space_provider)

Consumer already holds DataSpaceParticipant_jwt_vc_json from the Authority (2 VCs in wallet).
Provider already holds DataSpaceParticipant_jwt_vc_json from the Authority (2 VCs in wallet).


## 5. Mutual peer connection (`/api/v1/peer-connection/*` and `/api/v1/mates/all`)

Each side connects to the other's gate (`POST /api/v1/peer-connection/connect` ->
`/api/v1/gate/access`) presenting its credential, and receives an access token in return. Both
directions are needed: the Consumer calls the Provider (catalog, negotiation, transfer requests)
and the Provider calls the Consumer back (offers, agreements, transfer starts). Once connected,
each peer appears in the other's participant registry (`GET /api/v1/mates/all`).

In [5]:
def approved_token(source_url, target_did):
    """The token source_url holds for target_did, or None if not connected yet."""
    items = requests.get(f"{source_url}/api/v1/peer-connection/request/all", headers=json_header).json().get("items", [])
    for req in reversed(items):
        if req.get("participant_id") == target_did and req.get("status") == "Approved" and req.get("final_token"):
            return req["final_token"]
        if req.get("participant_id") == target_did and req.get("status") == "Rejected":
            raise RuntimeError(f"Connection to {target_did} was rejected.")
    return None


def connect_peer(source_name, source_url, target_name, target_gate_url, target_did):
    if approved_token(source_url, target_did):
        print(f"{source_name} is already connected to {target_name}.")
        return
    connect_body = {
        "id": target_did,
        "nick": target_name,
        "url": target_gate_url,
        "actions": ["talk"],
        "visibility": "Public",
        "auto": True,
    }
    res = requests.post(f"{source_url}/api/v1/peer-connection/connect", json=connect_body, headers=json_header)
    if res.status_code not in (200, 201, 204):
        raise RuntimeError(f"{source_name} could not connect to {target_name}: {res.status_code} - {res.text}")
    for _ in range(30):
        if approved_token(source_url, target_did):
            print(f"{source_name} connected to {target_name}.")
            return
        time.sleep(1)
    raise TimeoutError(f"Timeout waiting for {source_name} to connect to {target_name}")


connect_peer("Consumer", data_space_consumer, "provider", f"{internal_provider_url}/api/v1/gate/access", data_space_provider_participant_id)
connect_peer("Provider", data_space_provider, "consumer", f"{internal_consumer_url}/api/v1/gate/access", data_space_consumer_participant_id)

consumer_mates = requests.get(f"{data_space_consumer}/api/v1/mates/all", headers=json_header).json().get("items", [])
print(f"Known participants in Consumer (/api/v1/mates/all): {[m.get('nick') or m.get('participant_id')[:24] for m in consumer_mates]}")

Consumer is already connected to provider.
Provider is already connected to consumer.
Known participants in Consumer (/api/v1/mates/all): ['did:jwk:eyJlIjoiQVFBQiIs', 'did:web:127.0.0.1%3A1500']


## 6. Peer tokens and DSP version discovery (`/.well-known/dspace-version/*` and `/rpc/.well-known/*`)

The agents attach these tokens to DSP calls by themselves; they are printed here only to show
what onboarding produced:
- `data_space_provider_participant_token`: held by the Consumer, presented to the Provider.
- `data_space_consumer_participant_token`: held by the Provider, presented to the Consumer.

An agent's own DSP base URL comes from `GET /.well-known/dspace-version/2025-1`, while a peer's
DSP base URL can be resolved directly from the participant registry via the Well-Known RPC
`POST /rpc/.well-known/dspace-version/path`.

In [6]:
data_space_provider_participant_token = approved_token(data_space_consumer, data_space_provider_participant_id)
data_space_consumer_participant_token = approved_token(data_space_provider, data_space_consumer_participant_id)

version = show(requests.get(f"{data_space_consumer}/.well-known/dspace-version/2025-1", headers=json_header))
data_space_consumer_dsp_endpoint = data_space_consumer + version["path"]

peer_version_path = show(requests.post(
    f"{data_space_consumer}/rpc/.well-known/dspace-version/path",
    headers=json_header,
    json={"participant_id": data_space_provider_participant_id},
))
data_space_provider_dsp_endpoint = peer_version_path["path"]

# Marker read by the guards of the other notebooks
onboarded = True

print(f"\nProvider token (held by Consumer): {data_space_provider_participant_token}")
print(f"Consumer token (held by Provider): {data_space_consumer_participant_token}")
print(f"Consumer DSP address:              {data_space_consumer_dsp_endpoint}")
print(f"Provider DSP address (via RPC):    {data_space_provider_dsp_endpoint}")

HTTP 200 GET http://127.0.0.1:1100/.well-known/dspace-version/2025-1
{
  "binding": "HTTPS",
  "path": "/dsp/current",
  "version": "2025-1",
  "auth": {
    "protocol": "GNAP",
    "version": "1"
  },
  "identifierType": "did:jwk",
  "serviceId": "urn:dsp-service-id:75a37eee-5a0d-58dd-a208-86bf3a668a3f"
}
HTTP 200 POST http://127.0.0.1:1100/rpc/.well-known/dspace-version/path
{
  "path": "http://127.0.0.1:1200/dsp/current"
}

Provider token (held by Consumer): mVJTtNIhCEqfIa07VpEMQSCrqSu7U9CorRicDOBW-cs
Consumer token (held by Provider): wgPCugOaFXaWYrHr-m3Gn60iNVUxYIK0Z65HisOSfeA
Consumer DSP address:              http://127.0.0.1:1100/dsp/current
Provider DSP address (via RPC):    http://127.0.0.1:1200/dsp/current
